## Transformation Logic & Schema Validation
This notebook performs unit testing to ensure that our business transformations and source data schemas are correct before we process the full 110M record dataset.

In [0]:
import unittest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

spark = SparkSession.builder.getOrCreate()

# Path to the dataset for integration testing
sample_path = "/Volumes/ecommerce_analytics_dev/bronze_layer/landing_csvs/2019-Oct.csv"

print(f"✅ Transformation Test Environment Ready for path: {sample_path}")

## 1. Validating the 'High Value' Business Rule
We use dummy data to test the logic that flags transactions over $500 as high-value. This ensures our marketing segmentation logic is mathematically sound before applying it to millions of rows.

In [0]:
# Create a broader set of dummy data to show insights
dummy_data = [
    (101, 650.0), # High value
    (102, 150.0), # Low value
    (103, 500.0), # Boundary (should be False)
    (104, 500.01) # Boundary (should be True)
]
df_dummy = spark.createDataFrame(dummy_data, ["user_id", "price"])

# Apply Transformation
transformed_dummy = df_dummy.withColumn("is_high_value", when(col("price") > 500, True).otherwise(False))

# Logic Preview
print("📊 DUMMY DATA LOGIC PREVIEW:")
transformed_dummy.show()

class TestTransformationLogic(unittest.TestCase):
    def test_logic_threshold(self):
        """Domain 4.2: Verify the >500 business rule"""
        res = {row['user_id']: row['is_high_value'] for row in transformed_dummy.collect()}
        self.assertTrue(res[101], "650 should be high value")
        self.assertFalse(res[102], "150 should NOT be high value")
        self.assertFalse(res[103], "Exactly 500 should NOT be high value")

suite_dummy = unittest.TestLoader().loadTestsFromTestCase(TestTransformationLogic)
result_dummy = unittest.TextTestRunner(verbosity=1).run(suite_dummy)

## 2. Source Schema Integrity Check
We sample the actual landing zone files to verify that all required columns are present. This acts as an early warning system for schema drift in the raw 110M records.

In [0]:
# Read a small slice of the dataset to verify structure
df_original = spark.read.option("header", "true").csv(sample_path).limit(500)

# Data Insights
print("📊 ORIGINAL DATA SAMPLE INSIGHTS:")
print(f"Rows Sampled: {df_original.count()}")
print(f"Columns Found: {df_original.columns}")

class TestOriginalSchema(unittest.TestCase):
    def test_source_columns(self):
        """Domain 3.3: Verify schema inference for critical columns"""
        required_cols = ["event_time", "event_type", "product_id", "price", "user_id", "user_session"]
        for c in required_cols:
            self.assertIn(c, df_original.columns, f"CRITICAL: {c} is missing from the source CSV!")

suite_orig = unittest.TestLoader().loadTestsFromTestCase(TestOriginalSchema)
result_orig = unittest.TextTestRunner(verbosity=1).run(suite_orig)

## 3. Consolidated Test Execution Report
This cell summarizes all tests. If any logic check fails, we raise an exception to halt the automated pipeline, preventing downstream corruption.

In [0]:
all_passed = result_dummy.wasSuccessful() and result_orig.wasSuccessful()
final_status = "✅ LOGIC VALIDATED" if all_passed else "❌ LOGIC FAILURE"

print("-" * 50)
print(f"FINAL TRANSFORMATION SUMMARY: {final_status}")
print("-" * 50)
print(f"Dummy Logic Checks:  {'PASS' if result_dummy.wasSuccessful() else 'FAIL'}")
print(f"Source Schema Check: {'PASS' if result_orig.wasSuccessful() else 'FAIL'}")
print("-" * 50)

if not all_passed:
    raise Exception(f"Pipeline Automation Halted: {final_status}")